In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import re
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

# Φόρτωση BERT predictions
bert_valid = pd.read_csv('bert_valid_predictions.csv')
bert_test  = pd.read_csv('bert_test_predictions.csv')

y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

print(f'Train: {len(train)} δείγματα')
print(f'Valid: {len(valid)} δείγματα')
print(f'Test:  {len(test)} δείγματα')
print(f'\nBERT valid predictions: {len(bert_valid)}')
print(f'BERT test predictions:  {len(bert_test)}')

Train: 5082 δείγματα
Valid: 565 δείγματα
Test:  997 δείγματα

BERT valid predictions: 565
BERT test predictions:  997


In [3]:
def preprocess_text(text):
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product,
                       verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard,
                         average='macro', zero_division=0)
    y_true_hazard  = np.array(y_true_hazard)
    y_pred_hazard  = np.array(y_pred_hazard)
    y_true_product = np.array(y_true_product)
    y_pred_product = np.array(y_pred_product)
    
    correct_hazard_mask = (y_true_hazard == y_pred_hazard)
    n_correct = correct_hazard_mask.sum()
    
    if n_correct == 0:
        f1_product = 0.0
    else:
        f1_product = f1_score(
            y_true_product[correct_hazard_mask],
            y_pred_product[correct_hazard_mask],
            average='macro', zero_division=0
        )
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'Σωστά hazard predictions:           {n_correct}/{len(y_true_hazard)} ({100*n_correct/len(y_true_hazard):.1f}%)')
        print(f'macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score

In [4]:
# Προετοιμασία κειμένων
train['combined'] = train['title'].fillna('') + ' ' + train['text'].fillna('').str[:550]
valid['combined'] = valid['title'].fillna('') + ' ' + valid['text'].fillna('').str[:550]
test['combined']  = test['title'].fillna('')  + ' ' + test['text'].fillna('').str[:550]

train['combined'] = train['combined'].apply(preprocess_text)
valid['combined'] = valid['combined'].apply(preprocess_text)
test['combined']  = test['combined'].apply(preprocess_text)

# TF-IDF με βέλτιστες παραμέτρους
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    stop_words='english'
)

X_train = tfidf.fit_transform(train['combined'])
X_valid = tfidf.transform(valid['combined'])
X_test  = tfidf.transform(test['combined'])

# SVM classifiers
clf_svm_hazard = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_hazard.fit(X_train, train['hazard-category'])

clf_svm_product = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
clf_svm_product.fit(X_train, train['product-category'])

# SVM predictions
svm_valid_hazard  = clf_svm_hazard.predict(X_valid)
svm_valid_product = clf_svm_product.predict(X_valid)
svm_test_hazard   = clf_svm_hazard.predict(X_test)
svm_test_product  = clf_svm_product.predict(X_test)

print('=== SVM Score ===\n')
score_svm = official_st1_score(
    y_valid_hazard, svm_valid_hazard,
    y_valid_product, svm_valid_product
)

=== SVM Score ===

macro-F1 Hazard:                    0.8040
Σωστά hazard predictions:           526/565 (93.1%)
macro-F1 Product (given correct h): 0.6798
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.7419


In [5]:
print('=== BERT Score ===\n')
score_bert = official_st1_score(
    y_valid_hazard, bert_valid['hazard_bert'],
    y_valid_product, bert_valid['product_bert']
)

=== BERT Score ===

macro-F1 Hazard:                    0.8556
Σωστά hazard predictions:           523/565 (92.6%)
macro-F1 Product (given correct h): 0.7439
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
OFFICIAL ST1 SCORE:                 0.7998


In [6]:
def weighted_ensemble_predict(pred_A, pred_B, weight_A, weight_B):
    """
    Weighted voting μεταξύ δύο μοντέλων.
    """
    pred_A = np.array(pred_A)
    pred_B = np.array(pred_B)
    final_preds = []
    
    for i in range(len(pred_A)):
        if pred_A[i] == pred_B[i]:
            # Συμφωνούν — εύκολη απόφαση
            final_preds.append(pred_A[i])
        else:
            # Διαφωνούν — επιλέγουμε βάσει βάρους
            # Το μοντέλο με το μεγαλύτερο βάρος κερδίζει
            if weight_A >= weight_B:
                final_preds.append(pred_A[i])
            else:
                final_preds.append(pred_B[i])
    
    return np.array(final_preds)


# Δοκιμή διαφορετικών βαρών
weight_combinations = [
    (0.9, 0.1),
    (0.7, 0.3),
    (0.5, 0.5),
    (0.3, 0.7),
    (0.1, 0.9),
]

print('=== ENSEMBLE BERT + SVM ===')
print('(Βάρη: SVM, BERT)\n')

best_score  = 0
best_weights = None

for w_svm, w_bert in weight_combinations:
    ens_hazard  = weighted_ensemble_predict(
        svm_valid_hazard, bert_valid['hazard_bert'], w_svm, w_bert)
    ens_product = weighted_ensemble_predict(
        svm_valid_product, bert_valid['product_bert'], w_svm, w_bert)
    
    score = official_st1_score(
        y_valid_hazard, ens_hazard,
        y_valid_product, ens_product,
        verbose=False
    )
    print(f'SVM={w_svm}, BERT={w_bert} → Score: {score:.4f}')
    
    if score > best_score:
        best_score   = score
        best_weights = (w_svm, w_bert)

print(f'\nΒέλτιστα βάρη: SVM={best_weights[0]}, BERT={best_weights[1]}')
print(f'Βέλτιστο Score: {best_score:.4f}')
print(f'\nΣύγκριση:')
print(f'SVM μόνο:  0.7419')
print(f'BERT μόνο: 0.7998')
print(f'Ensemble:  {best_score:.4f}')

=== ENSEMBLE BERT + SVM ===
(Βάρη: SVM, BERT)

SVM=0.9, BERT=0.1 → Score: 0.7419
SVM=0.7, BERT=0.3 → Score: 0.7419
SVM=0.5, BERT=0.5 → Score: 0.7419
SVM=0.3, BERT=0.7 → Score: 0.7998
SVM=0.1, BERT=0.9 → Score: 0.7998

Βέλτιστα βάρη: SVM=0.3, BERT=0.7
Βέλτιστο Score: 0.7998

Σύγκριση:
SVM μόνο:  0.7419
BERT μόνο: 0.7998
Ensemble:  0.7998


In [7]:
# Παίρνουμε decision scores από το SVM
svm_hazard_scores  = clf_svm_hazard.decision_function(X_valid)
svm_product_scores = clf_svm_product.decision_function(X_valid)

# Normalize scores σε [0,1]
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
svm_hazard_scores_norm  = scaler.fit_transform(svm_hazard_scores)
svm_product_scores_norm = scaler.fit_transform(svm_product_scores)

print(f'SVM hazard scores shape:  {svm_hazard_scores_norm.shape}')
print(f'SVM product scores shape: {svm_product_scores_norm.shape}')
print(f'Hazard classes:  {clf_svm_hazard.classes_}')
print(f'Product classes: {clf_svm_product.classes_}')
print('SVM scores έτοιμα!')

SVM hazard scores shape:  (565, 10)
SVM product scores shape: (565, 22)
Hazard classes:  ['allergens' 'biological' 'chemical' 'food additives and flavourings'
 'foreign bodies' 'fraud' 'migration' 'organoleptic aspects'
 'other hazard' 'packaging defect']
Product classes: ['alcoholic beverages' 'cereals and bakery products'
 'cocoa and cocoa preparations, coffee and tea' 'confectionery'
 'dietetic foods, food supplements, fortified foods' 'fats and oils'
 'feed materials' 'food additives and flavourings'
 'food contact materials' 'fruits and vegetables' 'herbs and spices'
 'honey and royal jelly' 'ices and desserts'
 'meat, egg and dairy products' 'non-alcoholic beverages'
 'nuts, nut products and seeds' 'other food product / mixed' 'pet feed'
 'prepared dishes and snacks' 'seafood'
 'soups, broths, sauces and condiments' 'sugars and syrups']
SVM scores έτοιμα!


In [9]:
# Φόρτωση BERT probabilities
bert_valid_hazard_probs  = np.load('bert_valid_hazard_probs.npy')
bert_valid_product_probs = np.load('bert_valid_product_probs.npy')
bert_test_hazard_probs   = np.load('bert_test_hazard_probs.npy')
bert_test_product_probs  = np.load('bert_test_product_probs.npy')

print(f'BERT valid hazard probs:  {bert_valid_hazard_probs.shape}')
print(f'BERT valid product probs: {bert_valid_product_probs.shape}')
print(f'BERT test hazard probs:   {bert_test_hazard_probs.shape}')
print(f'BERT test product probs:  {bert_test_product_probs.shape}')

BERT valid hazard probs:  (565, 10)
BERT valid product probs: (565, 22)
BERT test hazard probs:   (997, 10)
BERT test product probs:  (997, 22)


In [10]:
from sklearn.preprocessing import MinMaxScaler

def probability_ensemble(svm_scores, bert_probs, svm_classes, w_svm, w_bert):
    """
    Συνδυάζει SVM scores και BERT probabilities.
    Για κάθε δείγμα: weighted_score = w_svm * svm + w_bert * bert
    Επιλέγει την κλάση με το μεγαλύτερο weighted score.
    """
    scaler = MinMaxScaler()
    svm_norm = scaler.fit_transform(svm_scores)
    
    combined = w_svm * svm_norm + w_bert * bert_probs
    pred_indices = combined.argmax(axis=1)
    return svm_classes[pred_indices]


# Δοκιμή διαφορετικών βαρών
weight_combinations = [
    (0.9, 0.1),
    (0.7, 0.3),
    (0.5, 0.5),
    (0.3, 0.7),
    (0.1, 0.9),
]

print('=== PROBABILITY ENSEMBLE BERT + SVM ===')
print('(Βάρη: SVM, BERT)\n')

best_score   = 0
best_weights = None

for w_svm, w_bert in weight_combinations:
    ens_hazard = probability_ensemble(
        svm_hazard_scores, bert_valid_hazard_probs,
        clf_svm_hazard.classes_, w_svm, w_bert
    )
    ens_product = probability_ensemble(
        svm_product_scores, bert_valid_product_probs,
        clf_svm_product.classes_, w_svm, w_bert
    )
    
    score = official_st1_score(
        y_valid_hazard, ens_hazard,
        y_valid_product, ens_product,
        verbose=False
    )
    print(f'SVM={w_svm}, BERT={w_bert} → Score: {score:.4f}')
    
    if score > best_score:
        best_score   = score
        best_weights = (w_svm, w_bert)

print(f'\nΒέλτιστα βάρη: SVM={best_weights[0]}, BERT={best_weights[1]}')
print(f'Βέλτιστο Score: {best_score:.4f}')
print(f'\nΣύγκριση:')
print(f'SVM μόνο:  0.7419')
print(f'BERT μόνο: 0.7998')
print(f'Ensemble:  {best_score:.4f}')

=== PROBABILITY ENSEMBLE BERT + SVM ===
(Βάρη: SVM, BERT)

SVM=0.9, BERT=0.1 → Score: 0.6655
SVM=0.7, BERT=0.3 → Score: 0.7402
SVM=0.5, BERT=0.5 → Score: 0.7876
SVM=0.3, BERT=0.7 → Score: 0.8028
SVM=0.1, BERT=0.9 → Score: 0.7998

Βέλτιστα βάρη: SVM=0.3, BERT=0.7
Βέλτιστο Score: 0.8028

Σύγκριση:
SVM μόνο:  0.7419
BERT μόνο: 0.7998
Ensemble:  0.8028


In [11]:
# Test scores από SVM
svm_test_hazard_scores  = clf_svm_hazard.decision_function(X_test)
svm_test_product_scores = clf_svm_product.decision_function(X_test)

# Ensemble predictions για test set
ensemble_test_hazard = probability_ensemble(
    svm_test_hazard_scores, bert_test_hazard_probs,
    clf_svm_hazard.classes_, 0.3, 0.7
)
ensemble_test_product = probability_ensemble(
    svm_test_product_scores, bert_test_product_probs,
    clf_svm_product.classes_, 0.3, 0.7
)

# Submission file
submission_ensemble = pd.DataFrame({
    'id': test['id'],
    'hazard-category': ensemble_test_hazard,
    'product-category': ensemble_test_product
})

submission_ensemble.to_csv('submission_ensemble_bert_svm.csv', index=False)
print('Submission file saved: submission_ensemble_bert_svm.csv')
print(f'Αριθμός predictions: {len(submission_ensemble)}')
print('\n--- Πρώτες 5 προβλέψεις ---')
print(submission_ensemble.head())

Submission file saved: submission_ensemble_bert_svm.csv
Αριθμός predictions: 997

--- Πρώτες 5 προβλέψεις ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological    prepared dishes and snacks
3   3      biological  meat, egg and dairy products
4   4  foreign bodies             ices and desserts


## Ensemble: Probability-based BERT + SVM

Συνδυασμός fine-tuned DistilBERT και TF-IDF+SVM με
weighted probability ensemble.

**Μέθοδος:**
Για κάθε δείγμα: `score = 0.3 * SVM_score + 0.7 * BERT_prob`
Επιλογή κλάσης με το μεγαλύτερο συνολικό score.

**Αποτελέσματα Validation:**
| Μοντέλο | Score |
|---|---|
| SVM μόνο | 0.7419 |
| BERT μόνο | 0.7998 |
| **Ensemble (0.3, 0.7)** | **0.8028** |

**Kaggle Score: 0.75465** — Best submission